This is one way to use generative AI. I am using GEN-AI to translate the following text:

Sandra and Paul are at a steak restaurant. A waiter greets them.

"Do you know what you would like to drink?" the waiter asks.
“Water and orange juice,” Sandra says.
"Thank you. Here are your menus," the waiter says.

The waiter brings water for Paul and orange juice for Sandra.

"What would you like to order?" the waiter asks.
"I would like a 12-ounce steak and mashed potatoes," Paul says.
"The same thing, but with green beans," Sandra says.
"And two orders of garlic bread," Paul says.
"Great. You should have it in soon," the waiter says.

The waiter returns after an hour.

“Sorry for your wait. Here are two orders of 12-ounce steaks with mashed potatoes and garlic bread,” the waiter says.
"I asked for green beans with mine," Sandra says.
"I'm sorry, I’ll get those for you," the waiter says.

The waiter quickly returns with Sandra's green beans.

In [4]:
import os
import ollama
from docx import Document

# --- CONFIGURATION ---
LOCAL_MODEL = "llama3" 

def translate_text(text):
    """Handles the connection with local Ollama and defines translation rules."""
    if not text.strip():
        return ""

    system_prompt = (
        "You are an expert translator. Translate the text from English to Spanish. "
        "Use a friendly tone with 'voseo'. Do not explain or greet. "
        "Only return the exact translation."
    )

    try:
        response = ollama.chat(model=LOCAL_MODEL, messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': f"Translate this: {text}"},
        ])
        return response['message']['content'].strip()
    except Exception as e:
        print(f"\nError in paragraph: {e}")
        return text

def translate_document(input_path, output_path):
    """Opens the .docx file, iterates through paragraphs and tables, and saves the result."""
    try:
        doc = Document(input_path)
    except Exception as e:
        print(f"File error: {e}")
        return

    total_paragraphs = len(doc.paragraphs)
    print(f"Translating {total_paragraphs} paragraphs with {LOCAL_MODEL}...")

    # Processes main text body
    for i, para in enumerate(doc.paragraphs):
        if para.text.strip():
            print(f"Processing {i+1}/{total_paragraphs}...", end="\r")
            para.text = translate_text(para.text)
    
    # Processes text inside tables
    print("\nProcessing tables...")
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                for paragraph in cell.paragraphs:
                    if paragraph.text.strip():
                        paragraph.text = translate_text(paragraph.text)

    doc.save(output_path)
    print("\nProcess completed.")

if __name__ == "__main__":
    original_file = r"text_to_translate.docx" 
    output_file = r"text_translated.docx"
    
    if os.path.exists(original_file):
        translate_document(original_file, output_file)
    else:
        print(f"ERROR: File not found at: {original_file}")

Translating 8 paragraphs with llama3...
Processing 7/8...
Processing tables...

Process completed.
